# Практика · ML Handbook гл. 35 (backprop)

Источник: [backpropagation](https://mlhandbook-production.up.railway.app/parts/deep-learning/backprop.html)

Теория: [`d.4.35 … .md`](../d.4.35%20Backpropagation/d.4.35%20Backpropagation.md) · [`README.md`](./README.md)

**Handbook-код — в этом файле** (§35.3 → §35.6). Deep-ML: [`deep_ml/`](./deep_ml/README.md).

| ☐ | Блок |
|---|------|
| ☐ | §35.3 XOR NumPy |
| ☐ | §35.4 XOR PyTorch |
| ☐ | §35.6 gradient check |


# §35.3 · XOR, backprop на NumPy

Чеклист: [`handbook_gl35_практика.ipynb`](./handbook_gl35_практика.ipynb)

Сеть **2→4→1**, sigmoid, MSE. Допиши **forward** и **backward** в цикле (как на сайте).

In [ ]:
import numpy as np


def need(cond, msg="допиши код выше"):
    assert cond, msg


def sigmoid(z):
  ...


rng = np.random.default_rng(0)
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
y = np.array([[0], [1], [1], [0]], float)

W1 = rng.normal(0, 1, (2, 4))
b1 = np.zeros((1, 4))
W2 = rng.normal(0, 1, (4, 1))
b2 = np.zeros((1, 1))
lr = 0.5

## Обучение (5000 эпох)

В каждой итерации: forward → backward → обновление `W1,b1,W2,b2`. Шаблон — [handbook §35.3](https://mlhandbook-production.up.railway.app/parts/deep-learning/backprop.html).

In [ ]:
for epoch in range(5000):
    # --- forward ---
    z1 = ...
    a1 = ...
    z2 = ...
    a2 = ...
    loss = ...
    # --- backward ---
    d_a2 = ...
    d_z2 = ...
    d_W2 = ...
    d_b2 = ...
    d_a1 = ...
    d_z1 = ...
    d_W1 = ...
    d_b1 = ...
    # --- шаг ---
    W2 -= lr * d_W2
    b2 -= lr * d_b2
    W1 -= lr * d_W1
    b1 -= lr * d_b1

pred = sigmoid(sigmoid(X @ W1 + b1) @ W2 + b2).round(2).ravel()
print("предсказания XOR:", pred)
print("правильно       :", y.ravel())
need(np.allclose(pred, [0.02, 0.95, 0.95, 0.06], atol=0.05))

---


# §35.4 · XOR, PyTorch + autograd

Чеклист: [`handbook_gl35_практика.ipynb`](./handbook_gl35_практика.ipynb)

Те же данные XOR. Петля: `zero_grad` → forward → `backward` → `step`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn


def need(cond, msg="допиши код выше"):
    assert cond, msg


X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
y = np.array([[0], [1], [1], [0]], float)
Xt = torch.tensor(X, dtype=torch.float32)
yt = torch.tensor(y, dtype=torch.float32)

In [ ]:
net = nn.Sequential(nn.Linear(2, 4), nn.Sigmoid(), nn.Linear(4, 1), nn.Sigmoid())
opt = torch.optim.Adam(net.parameters(), lr=0.1)
lossf = nn.MSELoss()

for epoch in range(2000):
    ...  # opt.zero_grad(); loss = lossf(net(Xt), yt); loss.backward(); opt.step()

out = net(Xt).detach().round().ravel().tolist()
print("предсказания XOR (torch):", out)
need(out == [0.0, 1.0, 1.0, 0.0])

---


# §35.6 · градиентная проверка

Чеклист: [`handbook_gl35_практика.ipynb`](./handbook_gl35_практика.ipynb)

Один вес `w`, лосс `(sigmoid(w*x) - y)²`. Сверка численного и аналитического ∂L/∂w.

In [ ]:
import numpy as np


def need(cond, msg="допиши код выше"):
    assert cond, msg


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def loss(w, x=2.0, y=1.0):
    a = sigmoid(w * x)
    return (a - y) ** 2


w = 0.5
h = 1e-5
num_grad = ...  # (loss(w+h) - loss(w-h)) / (2*h)
a = sigmoid(w * 2.0)
ana_grad = ...  # цепочка: 2*(a-y) * a*(1-a) * x

print(f"численный градиент: {num_grad:.6f}")
print(f"аналитический      : {ana_grad:.6f}")
print("совпадают:", np.isclose(num_grad, ana_grad))
need(np.isclose(num_grad, ana_grad))
need(abs(num_grad - (-0.211508)) < 1e-4)